# 02. Тест зоопарка моделей
Быстрая проверка, что все 6 архитектур (TCN-AE/VAE, LSTM-AE/VAE, Transformer-AE, гибрид TCN+Attention-VAE) собираются, делают forward-pass правильной формы и градиенты текут (backward без ошибок).

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd() / "notebooks").exists() is False and pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
# делаем import tsprep рабочим независимо от того, откуда запущен ноутбук
for cand in [pathlib.Path.cwd(), pathlib.Path.cwd().parent]:
    src = cand / "src"
    if src.exists():
        sys.path.insert(0, str(src))
        break

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.random.seed(42)


In [2]:
from tsprep.config import DataConfig, ModelConfig, ALL_MODEL_NAMES
from tsprep.models.autoencoder import build_model
from tsprep.mlcore.losses import compute_loss
import torch

data_cfg = DataConfig(window_len=48, n_features=1)
x = torch.randn(8, data_cfg.window_len, data_cfg.n_features)

for name in ALL_MODEL_NAMES:
    model_cfg = ModelConfig(model_name=name, latent_dim=16, hidden_dim=32,
                             tcn_levels=4, d_model=32, n_heads=4)
    model = build_model(model_cfg, n_features=data_cfg.n_features, window_len=data_cfg.window_len)
    model.train()
    x_hat, z, mu, logvar = model(x)
    loss, parts = compute_loss(x_hat, x, mu, logvar, model.variational, "mse", beta=0.1)
    loss.backward()
    n_params = sum(p.numel() for p in model.parameters())
    has_grad = all(p.grad is not None for p in model.parameters() if p.requires_grad)
    print(f"{name:16s} | x_hat={tuple(x_hat.shape)} z={tuple(z.shape)} "
          f"variational={model.variational} params={n_params:,} grads_ok={has_grad} loss={parts['loss']:.4f}")


tcn_ae           | x_hat=(8, 48, 1) z=(8, 16) variational=False params=49,489 grads_ok=True loss=1.0989
tcn_vae          | x_hat=(8, 48, 1) z=(8, 16) variational=True params=50,017 grads_ok=True loss=1.5571
lstm_ae          | x_hat=(8, 48, 1) z=(8, 16) variational=False params=27,505 grads_ok=True loss=0.9954
lstm_vae         | x_hat=(8, 48, 1) z=(8, 16) variational=True params=28,033 grads_ok=True loss=1.0057
transformer_ae   | x_hat=(8, 48, 1) z=(8, 16) variational=False params=51,985 grads_ok=True loss=1.1130
hybrid_vae       | x_hat=(8, 48, 1) z=(8, 16) variational=True params=75,458 grads_ok=True loss=1.3024
